# 07. Formal expert system

## Цель

Это **последний исследовательский notebook** проекта.

В `01–06` уже были зафиксированы:
- EDA и feature selection;
- baseline-модели;
- hierarchical binary architecture;
- calibration + ensemble;
- prototype explanation layer;
- frozen inference bundle;
- external transfer / NHANES stress-test.

В `07` мы **не обучаем ML заново**.

Задача этой части — оформить медицинскую логику как отдельный экспертный слой:

> **facts → rules → fired rules → conflicts → recommendations → physician-facing conclusion**

### Архитектурный принцип

ML отвечает за вероятностную оценку дефицитов.

Экспертная система отвечает за:
- медицинские evidence;
- различие `positive / negative / unknown`;
- противоречия ML ↔ clinical evidence;
- полноту клинических данных;
- рекомендации по дополнительным анализам;
- объяснение решения.

Clinical rules **не смешиваются с ML probability**, потому что в `04`
их оптимальный вес в вероятностном ансамбле оказался равен 0.


In [1]:
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Any, Callable
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)


## 1. Находим проект и загружаем frozen данные

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw" / "deficiency_anemia.csv").exists():
            return candidate

    raise FileNotFoundError(
        "Не найден корень проекта с data/raw/deficiency_anemia.csv"
    )


PROJECT_ROOT = find_project_root()

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "deficiency_anemia.csv"
OOF_PATH = PROJECT_ROOT / "artifacts" / "ensemble" / "ensemble_oof_predictions.csv"
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "expert_system"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(DATA_PATH)

if not OOF_PATH.exists():
    raise FileNotFoundError(OOF_PATH)

df = pd.read_csv(DATA_PATH)
oof = pd.read_csv(OOF_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Internal dataset:", df.shape)
print("Frozen OOF predictions:", oof.shape)


PROJECT_ROOT: /home/sinopah/meditron
Internal dataset: (840, 48)
Frozen OOF predictions: (840, 30)


## 2. Targets экспертного слоя

Используем те же 6 binary targets, что и в hierarchical ML architecture.


In [3]:
TARGETS = [
    "iron_deficiency",
    "B12_deficiency",
    "folate_deficiency",
    "B6_deficiency",
    "copper_deficiency",
    "inflammation_anemia",
]

TARGET_DISPLAY = {
    "iron_deficiency": "Iron deficiency",
    "B12_deficiency": "Vitamin B12 deficiency",
    "folate_deficiency": "Folate deficiency",
    "B6_deficiency": "Vitamin B6 deficiency",
    "copper_deficiency": "Copper deficiency",
    "inflammation_anemia": "Anemia of inflammation",
}


## 3. Объединяем лабораторные данные с frozen OOF probabilities

`07` не использует true target для принятия решения.

True labels остаются только для анализа качества и проверки экспертного слоя.


In [4]:
if "patient_id" not in df.columns:
    raise ValueError("В dataset нет patient_id")

if "patient_id" not in oof.columns:
    raise ValueError("В ensemble_oof_predictions.csv нет patient_id")

expert_df = df.merge(
    oof,
    on="patient_id",
    how="inner",
    suffixes=("", "__oof"),
    validate="one_to_one",
)

print("Merged expert dataset:", expert_df.shape)
assert len(expert_df) == len(df)


Merged expert dataset: (840, 77)


## 4. Автоматически находим frozen probability / prediction columns

Это делает notebook устойчивым к небольшим различиям в именовании колонок.


In [5]:
def find_column(columns, candidates):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None


PREDICTION_COLUMNS = {}

for target in TARGETS:
    prob_col = find_column(
        expert_df.columns,
        [
            f"{target}__prob",
            f"{target}_prob",
            f"{target}__probability",
            f"{target}_probability",
            f"{target}__ensemble_prob",
            f"{target}_ensemble_prob",
        ],
    )

    pred_col = find_column(
        expert_df.columns,
        [
            f"{target}__pred",
            f"{target}_pred",
            f"{target}__prediction",
            f"{target}_prediction",
        ],
    )

    if prob_col is None or pred_col is None:
        raise ValueError(
            f"Не удалось найти frozen probability/prediction columns для {target}.\n"
            f"Проверь названия в ensemble_oof_predictions.csv."
        )

    PREDICTION_COLUMNS[target] = {
        "prob": prob_col,
        "pred": pred_col,
    }

pd.DataFrame(PREDICTION_COLUMNS).T


,prob,pred
iron_deficiency,iron_deficiency__probability,iron_deficiency__prediction
B12_deficiency,B12_deficiency__probability,B12_deficiency__prediction
folate_deficiency,folate_deficiency__probability,folate_deficiency__prediction
B6_deficiency,B6_deficiency__probability,B6_deficiency__prediction
copper_deficiency,copper_deficiency__probability,copper_deficiency__prediction
inflammation_anemia,inflammation_anemia__probability,inflammation_anemia__prediction


## 5. Medical knowledge base — текущая версия

Эти значения отражают уже использованную в проекте clinical logic.

**Важно:** после консультации врача корректируется только knowledge base.
ML-модели, calibration и результаты `04–06` при этом не переобучаются.

### Принцип
- `supports` — клинические данные поддерживают состояние;
- `against` — есть сильное evidence против;
- `unknown` — данных недостаточно / показатель не позволяет уверенно исключить состояние.

`unknown ≠ negative`.


In [6]:
KNOWLEDGE = {
    "anemia": {
        "female_hb_lt": 120.0,
        "male_hb_lt": 130.0,
    },
    "iron": {
        "serum_iron_low": 10.7,
        "tibc_low": 46.0,
        "tibc_high": 78.0,
        "tsat_low": 17.8,
        "ferritin_low": 11.0,
        "ferritin_support": 30.0,
        "ret_he_low": 30.6,
    },
    "b12": {
        "deficiency_lt": 140.0,
    },
    "folate": {
        "deficiency_lt": 4.0,
        "borderline_high": 8.0,
    },
}

print(json.dumps(KNOWLEDGE, indent=2, ensure_ascii=False))


{
  "anemia": {
    "female_hb_lt": 120.0,
    "male_hb_lt": 130.0
  },
  "iron": {
    "serum_iron_low": 10.7,
    "tibc_low": 46.0,
    "tibc_high": 78.0,
    "tsat_low": 17.8,
    "ferritin_low": 11.0,
    "ferritin_support": 30.0,
    "ret_he_low": 30.6
  },
  "b12": {
    "deficiency_lt": 140.0
  },
  "folate": {
    "deficiency_lt": 4.0,
    "borderline_high": 8.0
  }
}


## 6. Базовые структуры экспертной системы

In [7]:
@dataclass
class Evidence:
    target: str
    feature: str
    value: Any
    state: str
    message: str
    strength: str = "supporting"


@dataclass
class FiredRule:
    rule_id: str
    target: str
    state: str
    message: str
    evidence: list[dict]


@dataclass
class ExpertConclusion:
    target: str
    ml_probability: float
    ml_prediction: int
    rule_state: str
    agreement: str
    confidence: float
    confidence_label: str
    evidence_for: list[str]
    evidence_against: list[str]
    unknown_evidence: list[str]
    recommended_next_tests: list[str]
    fired_rules: list[str]


## 7. Безопасное чтение лабораторного значения

In [8]:
def get_value(row, column):
    if column not in row.index:
        return None

    value = row[column]

    if pd.isna(value):
        return None

    try:
        return float(value)
    except (TypeError, ValueError):
        return value


def confidence_label(score: float) -> str:
    if score >= 0.85:
        return "high"
    if score >= 0.65:
        return "moderate"
    return "low"


def predicted_state_confidence(probability: float, prediction: int) -> float:
    return float(probability if prediction == 1 else 1.0 - probability)


## 8. Правила Iron deficiency

Здесь clinical evidence строится из ferritin, TSAT, serum iron, Ret-He и TIBC.

Не требуем, чтобы все показатели присутствовали.
Отсутствующий показатель становится `unknown`, а не evidence против диагноза.


In [9]:
def evaluate_iron(row):
    evidence = []
    fired = []

    ferritin = get_value(row, "ferritin")
    tsat = get_value(row, "TSAT")
    serum_iron = get_value(row, "serum_iron")
    ret_he = get_value(row, "Ret_He")
    tibc = get_value(row, "TIBC")

    if ferritin is None:
        evidence.append(Evidence(
            "iron_deficiency", "ferritin", None, "unknown",
            "Ferritin отсутствует."
        ))
    elif ferritin < KNOWLEDGE["iron"]["ferritin_low"]:
        evidence.append(Evidence(
            "iron_deficiency", "ferritin", ferritin, "supports",
            f"Ferritin {ferritin:.2f} < {KNOWLEDGE['iron']['ferritin_low']}.",
            "strong"
        ))
    elif ferritin < KNOWLEDGE["iron"]["ferritin_support"]:
        evidence.append(Evidence(
            "iron_deficiency", "ferritin", ferritin, "unknown",
            f"Ferritin {ferritin:.2f} находится в зоне, которая не исключает iron deficiency."
        ))
    else:
        evidence.append(Evidence(
            "iron_deficiency", "ferritin", ferritin, "against",
            f"Ferritin {ferritin:.2f} не снижен."
        ))

    checks = [
        ("TSAT", tsat, KNOWLEDGE["iron"]["tsat_low"], "<"),
        ("serum_iron", serum_iron, KNOWLEDGE["iron"]["serum_iron_low"], "<"),
        ("Ret_He", ret_he, KNOWLEDGE["iron"]["ret_he_low"], "<"),
    ]

    for feature, value, threshold, _ in checks:
        if value is None:
            evidence.append(Evidence(
                "iron_deficiency", feature, None, "unknown",
                f"{feature} отсутствует."
            ))
        elif value < threshold:
            evidence.append(Evidence(
                "iron_deficiency", feature, value, "supports",
                f"{feature} {value:.2f} ниже порога {threshold}.",
                "supporting"
            ))
        else:
            evidence.append(Evidence(
                "iron_deficiency", feature, value, "against",
                f"{feature} {value:.2f} не снижен."
            ))

    if tibc is None:
        evidence.append(Evidence(
            "iron_deficiency", "TIBC", None, "unknown",
            "TIBC отсутствует."
        ))
    elif tibc > KNOWLEDGE["iron"]["tibc_high"]:
        evidence.append(Evidence(
            "iron_deficiency", "TIBC", tibc, "supports",
            f"TIBC {tibc:.2f} повышен.",
            "supporting"
        ))
    elif tibc < KNOWLEDGE["iron"]["tibc_low"]:
        evidence.append(Evidence(
            "iron_deficiency", "TIBC", tibc, "against",
            f"TIBC {tibc:.2f} снижен и менее типичен для isolated iron deficiency."
        ))
    else:
        evidence.append(Evidence(
            "iron_deficiency", "TIBC", tibc, "unknown",
            f"TIBC {tibc:.2f} не даёт сильного самостоятельного evidence."
        ))

    support_n = sum(e.state == "supports" for e in evidence)
    against_n = sum(e.state == "against" for e in evidence)
    available_n = sum(e.state != "unknown" for e in evidence)

    if support_n >= 2:
        state = "supports"
        fired.append("IRON_MULTIPLE_SUPPORT")
    elif available_n >= 2 and support_n == 0 and against_n >= 2:
        state = "against"
        fired.append("IRON_MULTIPLE_AGAINST")
    else:
        state = "unknown"
        fired.append("IRON_INSUFFICIENT_OR_MIXED")

    return state, evidence, fired


## 9. Правила Vitamin B12 deficiency

In [10]:
def evaluate_b12(row):
    evidence = []
    fired = []

    b12 = get_value(row, "vitamin_B12")
    active_b12 = get_value(row, "active_B12")
    mma = get_value(row, "MMA")
    homocysteine = get_value(row, "homocysteine")

    if b12 is None:
        evidence.append(Evidence(
            "B12_deficiency", "vitamin_B12", None, "unknown",
            "Vitamin B12 отсутствует."
        ))
        state = "unknown"
        fired.append("B12_NO_TOTAL_B12")

    elif b12 < KNOWLEDGE["b12"]["deficiency_lt"]:
        evidence.append(Evidence(
            "B12_deficiency", "vitamin_B12", b12, "supports",
            f"Vitamin B12 {b12:.2f} < {KNOWLEDGE['b12']['deficiency_lt']}.",
            "strong"
        ))
        state = "supports"
        fired.append("B12_LOW_TOTAL")

    else:
        evidence.append(Evidence(
            "B12_deficiency", "vitamin_B12", b12, "unknown",
            f"Vitamin B12 {b12:.2f} не является достаточным evidence для исключения функционального дефицита."
        ))
        state = "unknown"
        fired.append("B12_TOTAL_NOT_LOW_BUT_NOT_EXCLUSION")

    for feature, value in [
        ("active_B12", active_b12),
        ("MMA", mma),
        ("homocysteine", homocysteine),
    ]:
        if value is None:
            evidence.append(Evidence(
                "B12_deficiency", feature, None, "unknown",
                f"{feature} отсутствует."
            ))
        else:
            evidence.append(Evidence(
                "B12_deficiency", feature, value, "unknown",
                f"{feature} доступен; количественная интерпретация требует валидированного порога."
            ))

    return state, evidence, fired


## 10. Правила Folate deficiency

In [11]:
def evaluate_folate(row):
    evidence = []
    fired = []

    folate = get_value(row, "folate")

    if folate is None:
        evidence.append(Evidence(
            "folate_deficiency", "folate", None, "unknown",
            "Folate отсутствует."
        ))
        state = "unknown"
        fired.append("FOLATE_MISSING")

    elif folate < KNOWLEDGE["folate"]["deficiency_lt"]:
        evidence.append(Evidence(
            "folate_deficiency", "folate", folate, "supports",
            f"Folate {folate:.2f} < {KNOWLEDGE['folate']['deficiency_lt']}.",
            "strong"
        ))
        state = "supports"
        fired.append("FOLATE_LOW")

    elif folate <= KNOWLEDGE["folate"]["borderline_high"]:
        evidence.append(Evidence(
            "folate_deficiency", "folate", folate, "unknown",
            f"Folate {folate:.2f} находится в пограничной зоне."
        ))
        state = "unknown"
        fired.append("FOLATE_BORDERLINE")

    else:
        evidence.append(Evidence(
            "folate_deficiency", "folate", folate, "against",
            f"Folate {folate:.2f} не снижен."
        ))
        state = "against"
        fired.append("FOLATE_NOT_LOW")

    return state, evidence, fired


## 11. Правила Anemia of inflammation

Правило намеренно консервативное.

Если ключевых маркеров мало, возвращаем `unknown`, а не positive/negative.


In [12]:
def anemia_rule(row):
    hb = get_value(row, "hemoglobin")
    sex = row.get("sex", None)

    if hb is None or pd.isna(sex):
        return None

    sex = str(sex).strip().upper()

    if sex in {"F", "FEMALE", "Ж", "ЖЕН"}:
        return int(hb < KNOWLEDGE["anemia"]["female_hb_lt"])

    if sex in {"M", "MALE", "М", "МУЖ"}:
        return int(hb < KNOWLEDGE["anemia"]["male_hb_lt"])

    return None


def evaluate_inflammation(row):
    evidence = []
    fired = []

    anemia = anemia_rule(row)
    serum_iron = get_value(row, "serum_iron")
    tibc = get_value(row, "TIBC")
    ferritin = get_value(row, "ferritin")
    crp = get_value(row, "CRP")

    if anemia == 0:
        evidence.append(Evidence(
            "inflammation_anemia", "hemoglobin", get_value(row, "hemoglobin"),
            "against",
            "Детерминированный anemia rule не подтверждает анемию.",
            "strong"
        ))
        return "against", evidence, ["INFLAMMATION_NO_ANEMIA"]

    if anemia is None:
        evidence.append(Evidence(
            "inflammation_anemia", "hemoglobin", None,
            "unknown",
            "Недостаточно данных для определения анемии."
        ))

    clinical_checks = []

    if serum_iron is not None:
        clinical_checks.append(serum_iron < KNOWLEDGE["iron"]["serum_iron_low"])
        evidence.append(Evidence(
            "inflammation_anemia", "serum_iron", serum_iron,
            "supports" if serum_iron < KNOWLEDGE["iron"]["serum_iron_low"] else "against",
            "Serum iron снижен." if serum_iron < KNOWLEDGE["iron"]["serum_iron_low"] else "Serum iron не снижен."
        ))
    else:
        evidence.append(Evidence(
            "inflammation_anemia", "serum_iron", None, "unknown",
            "Serum iron отсутствует."
        ))

    if tibc is not None:
        clinical_checks.append(tibc <= KNOWLEDGE["iron"]["tibc_high"])
        evidence.append(Evidence(
            "inflammation_anemia", "TIBC", tibc,
            "supports" if tibc <= KNOWLEDGE["iron"]["tibc_high"] else "against",
            "TIBC не повышен." if tibc <= KNOWLEDGE["iron"]["tibc_high"] else "TIBC повышен."
        ))
    else:
        evidence.append(Evidence(
            "inflammation_anemia", "TIBC", None, "unknown",
            "TIBC отсутствует."
        ))

    if ferritin is not None:
        clinical_checks.append(ferritin >= KNOWLEDGE["iron"]["ferritin_low"])
        evidence.append(Evidence(
            "inflammation_anemia", "ferritin", ferritin,
            "supports" if ferritin >= KNOWLEDGE["iron"]["ferritin_low"] else "against",
            "Ferritin не снижен." if ferritin >= KNOWLEDGE["iron"]["ferritin_low"] else "Ferritin снижен."
        ))
    else:
        evidence.append(Evidence(
            "inflammation_anemia", "ferritin", None, "unknown",
            "Ferritin отсутствует."
        ))

    if crp is None:
        evidence.append(Evidence(
            "inflammation_anemia", "CRP", None, "unknown",
            "CRP отсутствует."
        ))
    else:
        evidence.append(Evidence(
            "inflammation_anemia", "CRP", crp, "unknown",
            f"CRP={crp:.2f}; учитывается как contextual evidence, без жёсткого cutoff в текущей knowledge base."
        ))

    available_n = len(clinical_checks)
    support_n = sum(clinical_checks)

    if anemia == 1 and available_n >= 2 and support_n == available_n:
        state = "supports"
        fired.append("INFLAMMATION_PATTERN_SUPPORT")
    elif anemia == 1 and available_n >= 2 and support_n == 0:
        state = "against"
        fired.append("INFLAMMATION_PATTERN_AGAINST")
    else:
        state = "unknown"
        fired.append("INFLAMMATION_INCOMPLETE_OR_MIXED")

    return state, evidence, fired


## 12. B6 и Copper — консервативная логика

Для этих состояний в текущей knowledge base **не зашиваем выдуманные универсальные cutoffs**.

Если специализированный маркер отсутствует — `unknown`.

Если он есть — тоже не превращаем автоматически в диагноз без медицински подтверждённого порога.
После ответа врача этот блок можно будет уточнить отдельно.


In [13]:
def evaluate_b6(row):
    value = get_value(row, "vitamin_B6")

    if value is None:
        evidence = [Evidence(
            "B6_deficiency", "vitamin_B6", None, "unknown",
            "Vitamin B6 отсутствует."
        )]
        return "unknown", evidence, ["B6_NO_VALIDATED_RULE"]

    evidence = [Evidence(
        "B6_deficiency", "vitamin_B6", value, "unknown",
        f"Vitamin B6={value:.2f}; universal cutoff не зафиксирован в текущей knowledge base."
    )]

    return "unknown", evidence, ["B6_NO_VALIDATED_RULE"]


def evaluate_copper(row):
    copper = get_value(row, "copper")
    ceruloplasmin = get_value(row, "ceruloplasmin")

    evidence = []

    if copper is None:
        evidence.append(Evidence(
            "copper_deficiency", "copper", None, "unknown",
            "Copper отсутствует."
        ))
    else:
        evidence.append(Evidence(
            "copper_deficiency", "copper", copper, "unknown",
            f"Copper={copper:.2f}; universal diagnostic cutoff не зафиксирован."
        ))

    if ceruloplasmin is None:
        evidence.append(Evidence(
            "copper_deficiency", "ceruloplasmin", None, "unknown",
            "Ceruloplasmin отсутствует."
        ))
    else:
        evidence.append(Evidence(
            "copper_deficiency", "ceruloplasmin", ceruloplasmin, "unknown",
            f"Ceruloplasmin={ceruloplasmin:.2f}; используется как contextual evidence."
        ))

    return "unknown", evidence, ["COPPER_NO_VALIDATED_RULE"]


## 13. Registry правил

In [14]:
RULE_EVALUATORS = {
    "iron_deficiency": evaluate_iron,
    "B12_deficiency": evaluate_b12,
    "folate_deficiency": evaluate_folate,
    "B6_deficiency": evaluate_b6,
    "copper_deficiency": evaluate_copper,
    "inflammation_anemia": evaluate_inflammation,
}


## 14. Conflict logic

Сильным конфликтом считаем только ситуации:

- ML positive + clinical rule `against`;
- ML negative + clinical rule `supports`.

`unknown` **никогда не считается конфликтом**.


In [15]:
def agreement_state(ml_prediction: int, rule_state: str) -> str:
    if rule_state == "unknown":
        return "insufficient_clinical_evidence"

    if ml_prediction == 1 and rule_state == "supports":
        return "agree_positive"

    if ml_prediction == 0 and rule_state == "against":
        return "agree_negative"

    if ml_prediction == 1 and rule_state == "against":
        return "conflict_ml_positive_rule_against"

    if ml_prediction == 0 and rule_state == "supports":
        return "conflict_ml_negative_rule_supports"

    return "unclear"


## 15. Recommended next tests

In [16]:
def recommend_next_tests(target, row, rule_state, agreement):
    recommendations = []

    if target == "iron_deficiency":
        for feature in ["ferritin", "TSAT", "Ret_He", "sTfR"]:
            if get_value(row, feature) is None:
                recommendations.append(feature)

        if agreement.startswith("conflict_") and get_value(row, "CRP") is None:
            recommendations.append("CRP")

    elif target == "B12_deficiency":
        if get_value(row, "vitamin_B12") is None:
            recommendations.append("vitamin_B12")

        for feature in ["active_B12", "MMA", "homocysteine"]:
            if get_value(row, feature) is None:
                recommendations.append(feature)

    elif target == "folate_deficiency":
        if get_value(row, "folate") is None:
            recommendations.append("folate")

        if agreement.startswith("conflict_"):
            for feature in ["vitamin_B12", "homocysteine"]:
                if get_value(row, feature) is None:
                    recommendations.append(feature)

    elif target == "B6_deficiency":
        if get_value(row, "vitamin_B6") is None:
            recommendations.append("vitamin_B6")

    elif target == "copper_deficiency":
        for feature in ["copper", "ceruloplasmin"]:
            if get_value(row, feature) is None:
                recommendations.append(feature)

    elif target == "inflammation_anemia":
        for feature in ["CRP", "ferritin", "serum_iron", "TIBC", "TSAT", "sTfR"]:
            if get_value(row, feature) is None:
                recommendations.append(feature)

    # preserve order and remove duplicates
    return list(dict.fromkeys(recommendations))


## 16. Expert engine для одного пациента


In [17]:
def evaluate_target(row, target):
    prob_col = PREDICTION_COLUMNS[target]["prob"]
    pred_col = PREDICTION_COLUMNS[target]["pred"]

    ml_probability = float(row[prob_col])
    ml_prediction = int(row[pred_col])

    rule_state, evidence, fired_rules = RULE_EVALUATORS[target](row)

    agreement = agreement_state(
        ml_prediction,
        rule_state,
    )

    confidence = predicted_state_confidence(
        ml_probability,
        ml_prediction,
    )

    # Конфликт или недостаточная clinical evidence не переписывают
    # ML probability, но делают вывод более осторожным.
    if agreement.startswith("conflict_"):
        confidence = min(confidence, 0.64)
    elif agreement == "insufficient_clinical_evidence":
        confidence = min(confidence, 0.84)

    evidence_for = [
        e.message for e in evidence
        if e.state == "supports"
    ]

    evidence_against = [
        e.message for e in evidence
        if e.state == "against"
    ]

    unknown_evidence = [
        e.message for e in evidence
        if e.state == "unknown"
    ]

    recommended = recommend_next_tests(
        target,
        row,
        rule_state,
        agreement,
    )

    return ExpertConclusion(
        target=target,
        ml_probability=ml_probability,
        ml_prediction=ml_prediction,
        rule_state=rule_state,
        agreement=agreement,
        confidence=float(confidence),
        confidence_label=confidence_label(confidence),
        evidence_for=evidence_for,
        evidence_against=evidence_against,
        unknown_evidence=unknown_evidence,
        recommended_next_tests=recommended,
        fired_rules=fired_rules,
    )


def evaluate_patient(row):
    conclusions = {
        target: evaluate_target(row, target)
        for target in TARGETS
    }

    return conclusions


## 17. Проверяем экспертную систему на одном пациенте


In [18]:
example_row = expert_df.iloc[0]
example_result = evaluate_patient(example_row)

for target, result in example_result.items():
    print("\n", "=" * 80)
    print(TARGET_DISPLAY[target])
    print(json.dumps(
        asdict(result),
        ensure_ascii=False,
        indent=2,
    ))



Iron deficiency
{
  "target": "iron_deficiency",
  "ml_probability": 0.0008894488792803,
  "ml_prediction": 0,
  "rule_state": "unknown",
  "agreement": "insufficient_clinical_evidence",
  "confidence": 0.84,
  "confidence_label": "moderate",
  "evidence_for": [],
  "evidence_against": [
    "Ret_He 32.80 не снижен."
  ],
  "unknown_evidence": [
    "Ferritin отсутствует.",
    "TSAT отсутствует.",
    "serum_iron отсутствует.",
    "TIBC отсутствует."
  ],
  "recommended_next_tests": [
    "ferritin",
    "TSAT",
    "sTfR"
  ],
  "fired_rules": [
    "IRON_INSUFFICIENT_OR_MIXED"
  ]
}

Vitamin B12 deficiency
{
  "target": "B12_deficiency",
  "ml_probability": 0.8282356325249303,
  "ml_prediction": 1,
  "rule_state": "unknown",
  "agreement": "insufficient_clinical_evidence",
  "confidence": 0.8282356325249303,
  "confidence_label": "moderate",
  "evidence_for": [],
  "evidence_against": [],
  "unknown_evidence": [
    "Vitamin B12 257.00 не является достаточным evidence для исключен

## 18. Запускаем expert engine на всех 840 пациентах


In [19]:
expert_rows = []
payloads = []

for _, row in expert_df.iterrows():
    patient_id = row["patient_id"]
    patient_results = evaluate_patient(row)

    patient_payload = {
        "patient_id": (
            int(patient_id)
            if isinstance(patient_id, (int, np.integer, float, np.floating))
            and float(patient_id).is_integer()
            else patient_id
        ),
        "anemia_rule": anemia_rule(row),
        "deficiencies": {},
        "conflicts": [],
        "recommended_next_tests": [],
    }

    all_recommendations = []

    for target, result in patient_results.items():
        result_dict = asdict(result)

        expert_rows.append({
            "patient_id": patient_id,
            "target": target,
            "ml_probability": result.ml_probability,
            "ml_prediction": result.ml_prediction,
            "rule_state": result.rule_state,
            "agreement": result.agreement,
            "confidence": result.confidence,
            "confidence_label": result.confidence_label,
            "n_evidence_for": len(result.evidence_for),
            "n_evidence_against": len(result.evidence_against),
            "n_unknown_evidence": len(result.unknown_evidence),
            "recommended_next_tests": "|".join(result.recommended_next_tests),
            "fired_rules": "|".join(result.fired_rules),
        })

        patient_payload["deficiencies"][target] = result_dict

        if result.agreement.startswith("conflict_"):
            patient_payload["conflicts"].append({
                "target": target,
                "type": result.agreement,
            })

        all_recommendations.extend(
            result.recommended_next_tests
        )

    patient_payload["recommended_next_tests"] = list(
        dict.fromkeys(all_recommendations)
    )

    payloads.append(patient_payload)

expert_results = pd.DataFrame(expert_rows)

print("Expert target-level rows:", expert_results.shape)
display(expert_results.head(12))


Expert target-level rows: (5040, 13)


,patient_id,target,ml_probability,ml_prediction,rule_state,agreement,confidence,confidence_label,n_evidence_for,n_evidence_against,n_unknown_evidence,recommended_next_tests,fired_rules
0,P000001,iron_deficiency,0.000889,0,unknown,insufficient_clinical_evidence,0.840000,moderate,0,1,4,ferritin|TSAT|sTfR,IRON_INSUFFICIENT_OR_MIXED
1,P000001,B12_deficiency,0.828236,1,unknown,insufficient_clinical_evidence,0.828236,moderate,0,0,4,,B12_TOTAL_NOT_LOW_BUT_NOT_EXCLUSION
2,P000001,folate_deficiency,0.130243,0,supports,conflict_ml_negative_rule_supports,0.640000,low,1,0,0,,FOLATE_LOW
3,P000001,B6_deficiency,0.003322,0,unknown,insufficient_clinical_evidence,0.840000,moderate,0,0,1,vitamin_B6,B6_NO_VALIDATED_RULE
4,P000001,copper_deficiency,0.000654,0,unknown,insufficient_clinical_evidence,0.840000,moderate,0,0,2,copper|ceruloplasmin,COPPER_NO_VALIDATED_RULE
5,P000001,inflammation_anemia,0.000183,0,against,agree_negative,0.999817,high,0,1,0,ferritin|serum_iron|TIBC|TSAT|sTfR,INFLAMMATION_NO_ANEMIA
6,P000002,iron_deficiency,0.000054,0,against,agree_negative,0.999946,high,0,2,3,ferritin|Ret_He|sTfR,IRON_MULTIPLE_AGAINST
7,P000002,B12_deficiency,0.011015,0,unknown,insufficient_clinical_evidence,0.840000,moderate,0,0,4,vitamin_B12|active_B12|MMA|homocysteine,B12_NO_TOTAL_B12
8,P000002,folate_deficiency,0.081216,0,unknown,insufficient_clinical_evidence,0.840000,moderate,0,0,1,folate,FOLATE_MISSING
9,P000002,B6_deficiency,0.000748,0,unknown,insufficient_clinical_evidence,0.840000,moderate,0,0,1,vitamin_B6,B6_NO_VALIDATED_RULE


## 19. Сводка rule states

In [20]:
rule_state_summary = (
    expert_results
    .groupby(["target", "rule_state"])
    .size()
    .reset_index(name="n")
)

display(rule_state_summary)


,target,rule_state,n
0,B12_deficiency,supports,62
1,B12_deficiency,unknown,778
2,B6_deficiency,unknown,840
3,copper_deficiency,unknown,840
4,folate_deficiency,against,178
5,folate_deficiency,supports,130
6,folate_deficiency,unknown,532
7,inflammation_anemia,against,320
8,inflammation_anemia,supports,92
9,inflammation_anemia,unknown,428


## 20. Сводка ML ↔ rule agreement/conflicts

In [21]:
agreement_summary = (
    expert_results
    .groupby(["target", "agreement"])
    .size()
    .reset_index(name="n")
    .sort_values(["target", "n"], ascending=[True, False])
)

display(agreement_summary)


,target,agreement,n
1,B12_deficiency,insufficient_clinical_evidence,778
0,B12_deficiency,agree_positive,62
2,B6_deficiency,insufficient_clinical_evidence,840
3,copper_deficiency,insufficient_clinical_evidence,840
7,folate_deficiency,insufficient_clinical_evidence,532
4,folate_deficiency,agree_negative,178
5,folate_deficiency,agree_positive,112
6,folate_deficiency,conflict_ml_negative_rule_supports,18
12,inflammation_anemia,insufficient_clinical_evidence,428
8,inflammation_anemia,agree_negative,319


## 21. Только реальные conflicts

`unknown` сюда не попадает.


In [22]:
conflicts = expert_results[
    expert_results["agreement"].str.startswith("conflict_")
].copy()

conflict_summary = (
    conflicts
    .groupby(["target", "agreement"])
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)

display(conflict_summary)


,target,agreement,n
3,iron_deficiency,conflict_ml_negative_rule_supports,46
1,inflammation_anemia,conflict_ml_negative_rule_supports,40
0,folate_deficiency,conflict_ml_negative_rule_supports,18
4,iron_deficiency,conflict_ml_positive_rule_against,4
2,inflammation_anemia,conflict_ml_positive_rule_against,1


## 22. Confidence summary

In [23]:
confidence_summary = (
    expert_results["confidence_label"]
    .value_counts()
    .rename_axis("confidence_label")
    .reset_index(name="n")
)

confidence_summary["pct"] = (
    confidence_summary["n"]
    / len(expert_results)
    * 100
)

display(confidence_summary)


,confidence_label,n,pct
0,moderate,3691,73.234127
1,high,1181,23.432540
2,low,168,3.333333


## 23. Какие дополнительные исследования рекомендуются чаще всего

In [24]:
recommended_tests_flat = []

for payload in payloads:
    for test in payload["recommended_next_tests"]:
        recommended_tests_flat.append(test)

recommended_tests_summary = (
    pd.Series(recommended_tests_flat, dtype="object")
    .value_counts()
    .rename_axis("test")
    .reset_index(name="n")
)

display(recommended_tests_summary)


,test,n
0,copper,716
1,ceruloplasmin,709
2,vitamin_B6,705
3,Ret_He,614
4,sTfR,588
5,active_B12,588
6,MMA,568
7,homocysteine,542
8,folate,385
9,CRP,344


## 24. Physician-facing conclusion для одного пациента

Это прототип того, что позже сможет возвращать API.


In [25]:
def physician_summary(payload):
    lines = []

    lines.append(f"Patient ID: {payload['patient_id']}")
    lines.append(f"Anemia rule: {payload['anemia_rule']}")

    positive_targets = []

    for target, result in payload["deficiencies"].items():
        if result["ml_prediction"] == 1:
            positive_targets.append(
                (
                    target,
                    result["ml_probability"],
                    result["confidence_label"],
                    result["rule_state"],
                )
            )

    if positive_targets:
        lines.append("\nML-positive conditions:")

        for target, probability, confidence, rule_state in positive_targets:
            lines.append(
                f"- {TARGET_DISPLAY[target]}: "
                f"p={probability:.3f}, "
                f"confidence={confidence}, "
                f"clinical_rule={rule_state}"
            )
    else:
        lines.append("\nML-positive conditions: none")

    if payload["conflicts"]:
        lines.append("\nConflicts requiring review:")

        for conflict in payload["conflicts"]:
            lines.append(
                f"- {TARGET_DISPLAY[conflict['target']]}: "
                f"{conflict['type']}"
            )

    if payload["recommended_next_tests"]:
        lines.append(
            "\nRecommended next tests: "
            + ", ".join(payload["recommended_next_tests"])
        )

    lines.append(
        "\nInterpretation: decision-support output; "
        "clinical context and missing laboratory evidence must be considered."
    )

    return "\n".join(lines)


print(physician_summary(payloads[0]))


Patient ID: P000001
Anemia rule: 0

ML-positive conditions:
- Vitamin B12 deficiency: p=0.828, confidence=moderate, clinical_rule=unknown

Conflicts requiring review:
- Folate deficiency: conflict_ml_negative_rule_supports

Recommended next tests: ferritin, TSAT, sTfR, vitamin_B6, copper, ceruloplasmin, serum_iron, TIBC

Interpretation: decision-support output; clinical context and missing laboratory evidence must be considered.


## 25. Сохраняем expert-system artifacts


In [26]:
expert_results_path = OUTPUT_DIR / "expert_target_results.csv"
conflicts_path = OUTPUT_DIR / "expert_conflicts.csv"
recommendations_path = OUTPUT_DIR / "recommended_tests_summary.csv"
payloads_path = OUTPUT_DIR / "expert_payloads.jsonl"
knowledge_path = OUTPUT_DIR / "knowledge_base.json"

expert_results.to_csv(
    expert_results_path,
    index=False,
)

conflicts.to_csv(
    conflicts_path,
    index=False,
)

recommended_tests_summary.to_csv(
    recommendations_path,
    index=False,
)

with open(payloads_path, "w", encoding="utf-8") as f:
    for payload in payloads:
        f.write(
            json.dumps(
                payload,
                ensure_ascii=False,
                default=str,
            )
            + "\n"
        )

with open(knowledge_path, "w", encoding="utf-8") as f:
    json.dump(
        KNOWLEDGE,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Saved:", expert_results_path)
print("Saved:", conflicts_path)
print("Saved:", recommendations_path)
print("Saved:", payloads_path)
print("Saved:", knowledge_path)


Saved: /home/sinopah/meditron/artifacts/expert_system/expert_target_results.csv
Saved: /home/sinopah/meditron/artifacts/expert_system/expert_conflicts.csv
Saved: /home/sinopah/meditron/artifacts/expert_system/recommended_tests_summary.csv
Saved: /home/sinopah/meditron/artifacts/expert_system/expert_payloads.jsonl
Saved: /home/sinopah/meditron/artifacts/expert_system/knowledge_base.json


# Итог `07_expert_system`

После успешного выполнения notebook система имеет три независимых слоя:

### 1. ML layer
Calibrated L1 + CatBoost ensemble → probabilities.

### 2. Deterministic clinical layer
Anemia rule + target-specific clinical evidence.

### 3. Expert reasoning layer
- `supports / against / unknown`;
- ML ↔ rule agreement;
- conflicts;
- confidence;
- missing evidence;
- recommended next tests;
- physician-facing explanation.

## Почему это важно

Мы не пытаемся заменить ML правилами и не пытаемся искусственно улучшить probability клиническими правилами.

ML и expert knowledge выполняют разные функции:

> **ML predicts; expert system interprets and checks.**

После консультации врача медицинские thresholds / правила можно менять отдельно,
не переобучая ML.

---

# Что после `07`

Новых исследовательских notebooks не создаём.

Следующий этап — перенос готовой логики в production modules:

```text
ml/
    inference.py
    predictor.py
    clinical_rules.py

expert/
    facts.py
    knowledge_base.py
    engine.py
    explanations.py

api/
    ...
```

После этого делаем end-to-end API flow и frontend integration.


# Что прислать после запуска

Пришли мне:

1. `rule_state_summary`;
2. `agreement_summary`;
3. `conflict_summary`;
4. `confidence_summary`;
5. `recommended_tests_summary`;
6. пример `physician_summary`;
7. если будет ошибка — полный traceback.

После этого проверим, что экспертный слой ведёт себя логично, и `07` можно будет зафиксировать.
